This notebook fits by-cre models on the shendure data & demonstrates that estimated nb parameters track closely with UMI means, as expected.

The matrix-creation and fitting code is independent of the main scMPRA package, since its this analysis that informed the package code.

In [1]:
#imports
import pandas as pd
import numpy as np
import time
import pickle
from formulaic import Formula
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import linregress
from tensorzinb.tensorzinb import TensorZINB
import scMPRAforge as scm

import tqdm

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
data_root="/gpfs/gibbs/pi/reilly/tabula_data"
shendure=scm.scMPRA_data.from_tsv(f"{data_root}/shendure/shendure_counts_grouped.txt")

In [4]:
shendure.set_negative_controls(["minP","noP"])
shendure.set_reference_cell("Pluripotent")

In [5]:
shendure.ortho_filter()

scMPRAforge: INFO: Dropped 372 of 1833 (cell_type, cre_id) combos with fewer than 3 nonzero entries.


In [6]:
from dask.distributed import Client, LocalCluster
cluster=LocalCluster()
client = Client(cluster)

In [7]:
def _smart_matrix(data,split):
    """
    Takes data and split column & produces design matricies
    according to standard ortho schema.
    """
    anti=scm.anti_split(split)

    ## Decide model-type ##
    #0 levels will have already been filtered
    num_levels=len(data[anti].unique())
    if num_levels ==0:
        assert False, "Data were probably not filtered correctly."
    if num_levels>1:
        #more than one level between which we can perform hypothesis testing...
        model_type="contrastable"
    else:
        #only one level. Useless for hypothesis testing, but we will keep it around
        #in case we want to use it for simulation
        model_type="simulation_only"
    
    #now, pick the reference level
    if "reference" in data[anti].values:
        #if there is already a user-specificed reference level, let's just use that. 
        reference="reference"
    else:
        #Otherwise, use the most frequent level as the reference
        level_frequency=data[anti].value_counts()
        level_frequency=level_frequency.sort_values(ascending=False)
        reference=level_frequency.index[0]

    zi_formula="C(rep_id)-1"
    nb_formula=f"umis_mpra_bc ~ C({anti}, contr.treatment(base='{reference}'))"
    y, X=Formula(nb_formula).get_model_matrix(data,output='pandas')
    Z=Formula(zi_formula).get_model_matrix(data,output='pandas')
    return {
        'nb_regressors':X,
        'regressand':y,
        'zi_regressors':Z,
        'model_type':model_type,
        'nb_formula':nb_formula,
        'zi_formula':zi_formula,
        'reference':reference,
        'split':split
    }


def _tensorzinb_fit(matricies,name):
    """
    Takes matricies & produces a single tensorzinb model
    """
    zinbo = TensorZINB(endog=matricies["regressand"]["umis_mpra_bc"].to_numpy().squeeze(),
                    exog=matricies["nb_regressors"].to_numpy(),
                    exog_infl=matricies["zi_regressors"].to_numpy())
    

    return zinbo.fit(init_method="nb")

def standard_fit(data,split):
    """
    Takes an scMPRA object and produces a set of models along one axis,
    specified by split.
    """

    data=data.data
    levels=data[split].unique()

    mats_futures = {
        t: client.submit(
            _smart_matrix,
            data=data[data[split]==t],
            split=split
        )
        for t in levels
    }

    tzinb_futures = {
        t: client.submit(
                _tensorzinb_fit,
                mats_futures[t],
                t
            )
        for t in levels
    }
    
    return(mats_futures,
           scm.experiment_model(model=tzinb_futures,
                                split=split))

In [8]:
ct_mats,ct_model=standard_fit(shendure,"cell_type")

In [9]:
cre_mats,cre_model=standard_fit(shendure,"cre_id")

Metal device set to: Apple M4 Pro

systemMemory: 24.00 GB
maxCacheSize: 8.00 GB

Metal device set to: Apple M4 Pro

systemMemory: 24.00 GB
maxCacheSize: 8.00 GB

Metal device set to: Apple M4 Pro

systemMemory: 24.00 GB
maxCacheSize: 8.00 GB

Metal device set to: Apple M4 Pro

systemMemory: 24.00 GB
maxCacheSize: 8.00 GB

Metal device set to: Apple M4 Pro

systemMemory: 24.00 GB
maxCacheSize: 8.00 GB

Metal device set to: Apple M4 Pro

systemMemory: 24.00 GB
maxCacheSize: 8.00 GB

Metal device set to: Apple M4 Pro

systemMemory: 24.00 GB
maxCacheSize: 8.00 GB



In [10]:
scm.model_to_parameters(ct_model,ct_mats)
#ct_param=client.submit(scm.model_to_parameters,ct_model,ct_mats)

In [11]:
scm.model_to_parameters(cre_model,cre_mats)

In [ ]:
ct_param

In [ ]:
client.close()

In [ ]:
first_time=True
QC={}

if first_time:
    
    #loop through all cell-types, produce 
    #for cre_id in tqdm.tqdm(shendure.data["cre_id"].unique()):
    #for cre_id in tqdm.tqdm(list(shendure.data["cre_id"].unique()[0:2])+["eef1aP"]):
    
    split="cre_id"
    anti="cell_type"
    for level in ["Txndc12_chr4_7969"]:
    
        print(level)

        ## subset to current cell-type ##
        shendure_subset=shendure.data[shendure.data[split]==level]

        ## Decide model-type ##
        #0 levels will have already been filtered
        num_levels=len(shendure_subset[anti].unique())
        if num_levels ==0:
            assert False, "Data were probably not filtered correctly."
        if num_levels>1:
            #more than one level between which we can perform hypothesis testing...
            model_type="contrastable"
        else:
            #only one level. Useless for hypothesis testing, but we will keep it around
            #in case we want to use it for simulation
            model_type="simulation_only"
        
        #now, pick the reference level
        if "reference" in shendure_subset[anti].values:
            #if there is already a user-specificed reference level, let's just use that. 
            reference="reference"
        else:
            #Otherwise, use the most frequent level as the reference
            level_frequency=shendure_subset[anti].value_counts()
            level_frequency=level_frequency.sort_values(ascending=False)
            reference=level_frequency.index[0]

        
        ## create design matricies ##
        matricies=create_matricies(nb_formula=f"umis_mpra_bc ~ C(cell_type, contr.treatment(base='{reference}'))",
                            zi_formula="C(rep_id)-1",
                            scmpra=shendure_subset)
        
        
        ## take the mean for comparison ##
        shendure_mean=shendure_subset.groupby(anti)["umis_mpra_bc"].agg("mean").sort_values()

        ## fit a model##
        model=tensorzinb_fit(matricies)

        #if model=="index_error":
        #    print("index error")
        #    continue

        ## compute predictions of mu from the model ##
        linear_mu=matricies["nb_regressors"].to_numpy() @ model["weights"]["x_mu"]
        mu_predictions=np.exp(linear_mu)

        #compute predictions of ZI from the model
        #linear_zi=(matricies["zi_regressors"].to_numpy() @ model["weights"]["x_pi"])
        #zi_predictions=linear_zi=1/(1+np.exp(-linear_zi))

        ## get the names of the cell-types ##
        if model_type=="contrastable":
            cell_labeling=scm.undo_one_hot_encoding(matricies["nb_regressors"])
            cell_labeling=cell_labeling.rename({f"{anti}, contr.treatment(base='{reference}')":anti},axis=1)
            cell_labeling=cell_labeling[anti]
            cell_labeling=cell_labeling.str.removeprefix("T.")
        elif model_type=="simulation_only":
            ct=shendure_subset["cell_type"].unique().tolist()
            assert len(ct) ==1,"Somehow design matrix did not return cell_type despite multiple cell types in data"
            cell_labeling=pd.Series(np.repeat(ct[0],len(mu_predictions)))


        ## label the predictions with CRE names ##
        mu_predictions_df=pd.DataFrame({'cell_type':cell_labeling,'mu':mu_predictions.squeeze()})
        mu_predictions_df.sort_values(by='mu')

        ## merge means & predicted mu into one df for convienient plotting ##
        mu_summary = mu_predictions_df.groupby("cell_type").agg("mean")
        mu_summary = mu_summary.merge(shendure_mean.rename_axis('cell_type').reset_index(), on='cell_type', how='left')
        
        ### Regress & store data for plotting ###
        
        x = mu_summary["umis_mpra_bc"]
        y = mu_summary["mu"]

        # Fit regression
        try:
            slope, intercept, r_value, p_value, std_err = linregress(x, y)

            #store regression info
            ret={'x':x,
                'y':y,
                'model':model,
                'model_type':model_type,
                'slope':slope,
                'intercept':intercept,
                'r_value':r_value,
                'p_value':p_value,
                'std_err':std_err
            }
        except ValueError:
            print(f"regression error on {cre_id}")
            ret={'x':x,
                'y':y,
                'model':model,
                'model_type':model_type,
                'slope':None,
                'intercept':None,
                'r_value':None,
                'p_value':None,
                'std_err':None
            }
        
        QC[cre_id]=ret
    with open(f"{data_root}/shendure/cache_cre.pkl","wb") as f:
        pickle.dump(QC,f)
else:
    #not first time
    with open(f"{data_root}/shendure/cache_cre.pkl","rb") as f:
        QC=pickle.load(f)

Let's examine the min & max estimates for each cell-type to make sure they are in the right ballpark.

In [ ]:
for cell_type in QC:
    dat=QC[cell_type]["y"]
    print(f"{cell_type} : {min(dat)} to {max(dat)}")

All in the right ballpark!

In [ ]:
for cell_type in QC:
    print(f"{cell_type} : r={QC[cell_type]['r_value']}, slope={QC[cell_type]['slope']}")

Correlations look pretty good. There are a couple of unpleasant looking NANs. These are *probably* just instances where we only have one data point. Let's take a closer look.

In [ ]:
correlations={'cell_type':[],'r':[],'slope':[],'p':[]}

for cell_type in QC:
    c=QC[cell_type]#c for current
    correlations['cell_type'].append(cell_type)

Let's examine all of them graphically:

In [ ]:
for cell_type in QC:
    sns.regplot(x=QC[cell_type]["x"], y=QC[cell_type]["y"], scatter=True, line_kws={"color": "red"})
    plt.title(cell_type)
    plt.show()
    